<a href="https://colab.research.google.com/github/matsunagalab/lecture_ML/blob/main/machine_learning_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 第2回 事後学習: 機械学習の種類とラベルありデータ

この notebook は講義の**事後学習用**です。目安時間は **20分** 程度。

## 到達目標
1. **教師あり学習 / 教師なし学習 / 強化学習** の違いを説明できる
2. 教師あり学習の 2 タスク **回帰** と **分類(識別)** の違いを説明できる
3. この授業で使う代表的なラベルありデータ (California Housing / Iris / MNIST) の形・意味がわかる

## 進め方
- 上から順にセルを実行してください。
- コードよりも **各データの形 (shape) と意味** に注目してください。どの値が入力 $x$ で、どの値が出力 $y$ か、$y$ は実数か離散値か、を確認しながら進めます。


---
# 1. 復習: モデル $y = f(x \mid w)$

機械学習では、データから関数 $f$ のパラメータ $w$ を推定します。

| 記号     | 呼び方                                        |
|----------|-----------------------------------------------|
| $y$      | 出力・ラベル・目的変数                          |
| $x$      | 入力・説明変数・特徴量                          |
| $w$      | パラメータ (機械学習で推定する対象)            |

$N$ 個のデータはまとめて $\{(x_n, y_n)\}_{n=1}^{N}$ と書きます。


---
# 2. 機械学習の種類

データの与えられ方や目的によって大きく 3 つに分かれます。

| 種類         | データ               | 目的                              | 例                               |
|--------------|----------------------|-----------------------------------|----------------------------------|
| 教師あり学習 | $(x_n, y_n)$ のペア (ラベルあり) | 入力と出力の関係を学習            | 回帰 / 分類                       |
| 教師なし学習 | $x_n$ のみ (ラベルなし)        | データの構造を学習                | クラスタリング                    |
| 強化学習     | 環境との相互作用で得られる報酬 | 報酬を最大化する行動を学習        | ゲーム AI, 自転車に乗る           |

> この講義では **教師あり学習** と **教師なし学習** を扱います。強化学習はそれ自体で一分野を成すので扱いません。

## 教師あり学習の 2 タスク

| タスク   | 出力 $y$ の型 | 例                                |
|----------|----------------|-----------------------------------|
| 回帰     | 実数           | 住宅価格、株価、材料の硬さ       |
| 分類(識別) | 離散値 (カテゴリ) | 画像認識、文字認識、迷惑メール判定 |


---
# 3. この授業で扱うラベルありデータ

scikit-learn には学習・評価用の定番データセットが同梱されています。以下の 3 つは本講義でも何度も登場します。まずはそれぞれ **読み込んで shape を眺める** ことから始めましょう。


In [ ]:
# 共通で使うライブラリ
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets


## 3.1 California Housing (回帰の例)

米国カリフォルニア州のブロックごとの住宅価格データです。

- サンプル数: 20640
- 特徴量 $x$: 8 個 (中央所得、築年数、部屋数、緯度経度 など、すべて**実数**)
- 目的変数 $y$: **住宅価格 (実数)** → 回帰タスク


In [ ]:
# California Housing データセットをロード
california = datasets.fetch_california_housing()
print("x の shape :", california.data.shape)   # (20640, 8)
print("y の shape :", california.target.shape)  # (20640,)
print("特徴量名    :", california.feature_names)


In [ ]:
# y (住宅価格) は連続値 → 回帰タスクであることを確認
# 所得 (MedInc) と住宅価格の関係を散布図で見てみる
plt.figure(figsize=(6, 4))
plt.scatter(california.data[:, 0], california.target, c='k', s=5, alpha=0.3)
plt.xlabel("MedInc (中央所得)")
plt.ylabel("House value (目的変数 y)")
plt.title("California Housing: 所得と住宅価格")
plt.show()


## 3.2 Iris (アヤメ): 多クラス分類の例

統計学の父 R. A. Fisher が扱ったデータ。アヤメ 150 個体について 4 つの特徴量を計測し、**3 種**(setosa / versicolor / virginica) にラベル付けされています。

- サンプル数: 150
- 特徴量 $x$: 4 個 (がくの長さ・幅、花弁の長さ・幅)
- 目的変数 $y$: **3 種のいずれか (0/1/2 の離散値)** → 分類タスク


In [ ]:
# Iris データセットをロード
iris = datasets.load_iris()
print("x の shape :", iris.data.shape)    # (150, 4)
print("y の shape :", iris.target.shape)  # (150,)
print("特徴量名    :", iris.feature_names)
print("クラス名    :", iris.target_names)
print("y の値 (最初の 10 個):", iris.target[:10])  # 離散値であることに注目


In [ ]:
# がくの長さ vs がくの幅 を種ごとに色分けして散布図に
plt.figure(figsize=(6, 4))
for cls, color in zip([0, 1, 2], ['r', 'g', 'b']):
    mask = iris.target == cls
    plt.scatter(iris.data[mask, 0], iris.data[mask, 1],
                c=color, edgecolor='k', s=40, label=iris.target_names[cls])
plt.xlabel("sepal length (cm)")
plt.ylabel("sepal width (cm)")
plt.legend()
plt.title("Iris: 種ごとの散布図")
plt.show()


## 3.3 MNIST: 画像分類の例

手書き数字 0 〜 9 の画像データ。scikit-learn に同梱されている版は 8×8 画素の低解像度版です (オリジナルは 28×28)。

- サンプル数: 約 1800
- 特徴量 $x$: 64 個 (8×8 画素を並べたもの)
- 目的変数 $y$: **0 〜 9 の数字 (離散値)** → 10 クラス分類


In [ ]:
# MNIST (scikit-learn 版) をロード
mnist = datasets.load_digits()
print("x の shape      :", mnist.data.shape)    # (1797, 64)
print("画像 1 枚の shape :", mnist.images[0].shape)  # (8, 8)
print("y の shape      :", mnist.target.shape)  # (1797,)
print("y の値 (最初の 10 個):", mnist.target[:10])


In [ ]:
# 画像を 25 枚並べて表示
fig, axes = plt.subplots(5, 5, figsize=(6, 6),
                         subplot_kw={'xticks': [], 'yticks': []},
                         gridspec_kw=dict(hspace=0.4, wspace=0.3))
for i, ax in enumerate(axes.flat):
    ax.imshow(mnist.images[i], cmap='gray', interpolation='nearest')
    ax.set_title(mnist.target[i])
plt.show()


---
## 演習 2-1

上で扱った 3 つのデータセット (California Housing / Iris / MNIST) について、

- 入力 $x$ の次元 (特徴量の数)
- 出力 $y$ が **実数 (回帰)** か **離散値 (分類)** か

をそれぞれ答えてみましょう。次回以降の授業では、それぞれのタスクに応じた手法 (回帰・分類・クラスタリング) を詳しく学んでいきます。


**解答例 (スクロールする前にまず自分で考えてみてください)**

| データ            | $x$ の次元 | $y$ の型            | タスク     |
|-------------------|------------|---------------------|-----------|
| California Housing| 8          | 実数 (住宅価格)     | 回帰       |
| Iris              | 4          | 離散 (3 クラス)     | 分類(識別) |
| MNIST             | 64 (=8×8)  | 離散 (10 クラス)    | 分類(識別) |


---
# 4. コラム: モデル関数 $f(x|w)$ にはどんな種類があるか？

授業スライドで紹介した 3 種類をまとめておきます (詳細は今後の回で扱います)。

| モデル      | 式                                            | 特徴                                   |
|-------------|------------------------------------------------|----------------------------------------|
| 線形モデル  | $y = \sum_j w_j \phi_j(x)$                     | 解釈しやすい / 学習が簡単 / 次回から使う |
| カーネルモデル | $y = \sum_n w_n K(x, x_n)$                   | 関数形を明示しなくてよい / 高次元に強い |
| 非線形モデル | $y = f(x \mid w)$ (パラメータに非線形)         | 表現力が高い / ニューラルネットで使う   |

次回は**線形モデル**の最もシンプルな形である **線形回帰** を扱います。


---
# まとめ

- 機械学習は **教師あり / 教師なし / 強化学習** に分かれる。本講義は最初の 2 つを扱う。
- 教師あり学習は、$y$ が実数なら **回帰**、離散なら **分類(識別)**。
- 本講義で何度も登場するラベルありデータ: **California Housing (回帰), Iris (分類), MNIST (分類)**。
- モデル関数には **線形・カーネル・非線形** の 3 タイプがある。次回は線形回帰から。
